In [2]:
import os
#import multiprocessing

import polars as pl
import time

**Checking how powerfull this PC is**

In [ ]:
# Logical processors (Threads) - This is what Polars uses
threads = os.cpu_count()

print(f"🧠 CPU Cores/Threads available: {threads}")
print("-" * 30)

if threads <= 4:
    print("Verdict: Standard Laptop. Polars will run 4 chunks in parallel.")
elif threads >= 8:
    print("Verdict: Strong Machine! Polars will fly.")

🧠 CPU Cores/Threads available: 4
------------------------------
Verdict: Standard Laptop. Polars will run 4 chunks in parallel.


**Converting CSVs to Parquets**

In [3]:
csv_source = "D:/full_news/nasdaq_external_data.csv"
parquet_source = "D:/full_news/nasdaq_external_data.parquet"

In [ ]:
def convert_csv_to_parquet(csv_source: str, parquet_dest: str):
    # 1. Clean up partial/failed files from previous crashes
    if os.path.exists(parquet_dest):
        try:
            # Check if it's a valid parquet file (if it crashes, it's corrupt)
            pl.scan_parquet(parquet_dest).head(1).collect()
            print(f"⚠️ Parquet file already exists and is valid: {parquet_dest}")
            return
        except:
            print(f"♻️ Found partial/corrupt file at {parquet_dest}. Deleting and retrying...")
            os.remove(parquet_dest)

    print(f"Reading from: {csv_source}")
    print(f"Saving to: {parquet_dest}")
    print("🚀 Starting conversion... this will take a few minutes.")
    start_time = time.time()

    lazy_df = (
        pl.scan_csv(csv_source, try_parse_dates=False, ignore_errors=True)
        .with_columns(
            pl.col("Date").str.strptime(pl.Datetime(time_zone="UTC"), format="%Y-%m-%d %H:%M:%S UTC", strict=False)
        )
    )

    # FIX IS HERE: row_group_size=100000 reduces RAM usage significantly
    lazy_df.sink_parquet(
        parquet_dest, 
        row_group_size=50_000, 
        statistics=False
    )

    end_time = time.time()
    print(f"✅ Done! Converted in {(end_time - start_time)/60:.2f} minutes.")

convert_csv_to_parquet(csv_source, parquet_source)

Reading from: D:/full_news/nasdaq_external_data.csv
Saving to: D:/full_news/nasdaq_external_data.parquet
🚀 Starting conversion... this will take a few minutes.


: 

**Checking on conversion**
- whether it succesfully converted dates to datetime format without any malforming.
Fistly manually comparing the rows and afterwards mechanically.

In [12]:
csv = pl.scan_csv(csv_source).head(300).with_row_index("idx").collect()
parq = pl.scan_parquet(parquet_source).head(300).with_row_index("idx").collect()

In [13]:
def conversion_check_summary(csv, parq):
    print("Summary of Conversion Check:")
    print("-" * 30)
    total_csv = len(csv)
    total_parq = len(parq)

    print(f"Rows in CSV: {total_csv}")
    print(f"Rows in Parquet: {total_parq}")

    #One last check - Count how many nulls exist
    print(f"Null dates in CSV: {csv['Date'].null_count()}")
    print(f"Null dates in parquet: {parq['Date'].null_count()}")

    # Show the schema to confirm the type
    print(f"\nSchema:\n{parq.schema}")
    # Display first few rows
    with pl.Config(fmt_str_lengths=50, tbl_rows=10):
        display(parq)

conversion_check_summary(csv, parq)

Summary of Conversion Check:
------------------------------
Rows in CSV: 300
Rows in Parquet: 300
Null dates in CSV: 0
Null dates in parquet: 0

Schema:
Schema({'idx': UInt32, 'Date': Datetime(time_unit='us', time_zone='UTC'), 'Article_title': String, 'Stock_symbol': String, 'Url': String, 'Publisher': String, 'Author': String, 'Article': String, 'Lsa_summary': String, 'Luhn_summary': String, 'Textrank_summary': String, 'Lexrank_summary': String})


idx,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary
u32,"datetime[μs, UTC]",str,str,str,str,str,str,str,str,str,str
0,2020-06-05 06:30:54 UTC,"""Stocks That Hit 52-Week Highs On Friday""","""A""","""https://www.benzinga.com/news/20/06/16190091/stock…","""Benzinga Insights""",null,null,null,null,null,null
1,2020-06-03 06:45:20 UTC,"""Stocks That Hit 52-Week Highs On Wednesday""","""A""","""https://www.benzinga.com/news/20/06/16170189/stock…","""Benzinga Insights""",null,null,null,null,null,null
2,2020-05-26 00:30:07 UTC,"""71 Biggest Movers From Friday""","""A""","""https://www.benzinga.com/news/20/05/16103463/71-bi…","""Lisa Levin""",null,null,null,null,null,null
3,2020-05-22 08:45:06 UTC,"""46 Stocks Moving In Friday's Mid-Day Session""","""A""","""https://www.benzinga.com/news/20/05/16095921/46-st…","""Lisa Levin""",null,null,null,null,null,null
4,2020-05-22 07:38:59 UTC,"""B of A Securities Maintains Neutral on Agilent Tec…","""A""","""https://www.benzinga.com/news/20/05/16095304/b-of-…","""Vick Meyer""",null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…
295,2016-12-06 00:00:00 UTC,"""Pressure Biosciences CEO On Its Beaten Down Stock:…","""A""","""https://www.benzinga.com/general/biotech/16/12/877…","""Javier Hasse""",null,null,null,null,null,null
296,2016-11-16 00:00:00 UTC,"""The Market In 5 Minutes: DryShips, Trump And Targe…","""A""","""https://www.benzinga.com/news/16/11/8706577/the-ma…","""Benzinga News Desk""",null,null,null,null,null,null
297,2016-11-16 00:00:00 UTC,"""A Peek Into The Markets: U.S. Stock Futures Declin…","""A""","""https://www.benzinga.com/news/earnings/16/11/87066…","""Lisa Levin""",null,null,null,null,null,null


In [14]:
def conversion_check_manual(csv, parq):   
    # DIAGNOSTIC: Check what the Date column looks like in the original CSV
    print("Sample from CSV (original):")
    print(csv.select(["Date", "Stock_symbol"]).head(5))
    print(f"\nDate column type in CSV: {csv['Date'].dtype}")

    # Now check the Parquet file
    print("\nSample from Parquet (converted):")
    print(parq.select(["Date", "Stock_symbol"]).head(5))
    print(f"\nDate column type in Parquet: {parq['Date'].dtype}")

conversion_check_manual(csv, parq)

Sample from CSV (original):
shape: (5, 2)
┌─────────────────────────┬──────────────┐
│ Date                    ┆ Stock_symbol │
│ ---                     ┆ ---          │
│ str                     ┆ str          │
╞═════════════════════════╪══════════════╡
│ 2020-06-05 06:30:54 UTC ┆ A            │
│ 2020-06-03 06:45:20 UTC ┆ A            │
│ 2020-05-26 00:30:07 UTC ┆ A            │
│ 2020-05-22 08:45:06 UTC ┆ A            │
│ 2020-05-22 07:38:59 UTC ┆ A            │
└─────────────────────────┴──────────────┘

Date column type in CSV: String

Sample from Parquet (converted):
shape: (5, 2)
┌─────────────────────────┬──────────────┐
│ Date                    ┆ Stock_symbol │
│ ---                     ┆ ---          │
│ datetime[μs, UTC]       ┆ str          │
╞═════════════════════════╪══════════════╡
│ 2020-06-05 06:30:54 UTC ┆ A            │
│ 2020-06-03 06:45:20 UTC ┆ A            │
│ 2020-05-26 00:30:07 UTC ┆ A            │
│ 2020-05-22 08:45:06 UTC ┆ A            │
│ 2020-05-22 07:3

In [15]:
def conversion_check_automated(csv, parq):
    parq = parq.with_columns(
        pl.col("Date").dt.strftime("%Y-%m-%d %H:%M:%S UTC").alias("Date")
    )

    comparison = (
        csv.select(["idx", "Date"])
        .join(parq.select(["idx", "Date"]).rename({"Date": "Date_parquet"}), on="idx")
        .with_columns((pl.col("Date") == pl.col("Date_parquet")).alias("match"))
    )

    total = len(comparison)
    mismatches = (comparison["match"] == False).sum()

    print(f"Total rows compared: {total}")
    print(f"Mismatches: {mismatches}")

    if mismatches > 0:
        display(comparison.filter(~pl.col("match")).head(20))

conversion_check_automated(csv, parq)

Total rows compared: 300
Mismatches: 0


- This activity could lead to the leakage of future information
into the training set. Time series cross-validation (TSCV) was used to train and validate the model with folds (n = 5) on the news dataset. 
- Logistic Regression (LR) was selected for this study, among
other machine learning models, because of its simplicity, interpretability, and effectiveness in binary classification tasks and alignment with NGX stock index label categorization.
- in this study they used train data, validation data and test data. why?
-they used F1 score as evaluation metrcis for LR above accuracy
- (L2 Normalization) of TF-IDF vectors - you should use it i guess - Standard TF-IDF libraries like Scikit-Learn do this by default (double check it)